ONE HOT ENCODING

In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# Original data
df = pd.DataFrame({
    'color': ['red', 'green', 'blue', 'black', 'red']
})
print(df.head())
# Create and fit encoder
encoder = OneHotEncoder()
encoded = encoder.fit_transform(df[['color']]).toarray()

# DataFrame of encoded original data
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out())
print(encoded_df.head())
# Transform new data
e = encoder.transform([['blue']]).toarray()
encoded1_df = pd.DataFrame(e, columns=encoder.get_feature_names_out())
print(encoded1_df.head())

   color
0    red
1  green
2   blue
3  black
4    red
   color_black  color_blue  color_green  color_red
0          0.0         0.0          0.0        1.0
1          0.0         0.0          1.0        0.0
2          0.0         1.0          0.0        0.0
3          1.0         0.0          0.0        0.0
4          0.0         0.0          0.0        1.0
   color_black  color_blue  color_green  color_red
0          0.0         1.0          0.0        0.0


C:\Users\Varshitha\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [ ]:
# Concatenate original + new
encoded_all = pd.concat([encoded_df, encoded1_df], ignore_index=True)
#If you want the new row to continue with index 5 
# (since your original DataFrame has indices 0–4), 
# you just need to use ignore_index=False
encoded_all

,color_black,color_blue,color_green,color_red
0,0.0,0.0,0.0,1.0
1,0.0,0.0,1.0,0.0
2,0.0,1.0,0.0,0.0
3,1.0,0.0,0.0,0.0
4,0.0,0.0,0.0,1.0
0,0.0,1.0,0.0,0.0


In [5]:
# Extend df with new row
df_new = pd.concat([df, pd.DataFrame({'color': ['blue']})], ignore_index=True)

# Concatenate horizontally (indexes aligned now)
final_df = pd.concat([df_new, encoded_all], axis=1)

print(final_df)


   color  color_black  color_blue  color_green  color_red
0    red          0.0         0.0          0.0        1.0
1  green          0.0         0.0          1.0        0.0
2   blue          0.0         1.0          0.0        0.0
3  black          1.0         0.0          0.0        0.0
4    red          0.0         0.0          0.0        1.0
5   blue          0.0         1.0          0.0        0.0


LABEL ENCODING
~assigning unique numerical label to each category in the variable in an alphabetical order or based on frequency of categories(rank).
red:1
green:2
blue:3

In [85]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
le.fit(['paris','tokyo','tokyo'])
print(le.classes_)
print(le.transform(['paris','tokyo']))
le.inverse_transform([0,1,1,1])

['paris' 'tokyo']
[0 1]


array(['paris', 'tokyo', 'tokyo', 'tokyo'], dtype='<U5')

In [86]:
df1 = pd.DataFrame({
    'color': ['red', 'green', 'blue', 'black', 'red']
})
df1['color_encoded'] = le.fit_transform(df1['color'])
#since we need only 1 column not array
print(df1)
le.classes_

   color  color_encoded
0    red              3
1  green              2
2   blue              1
3  black              0
4    red              3


array(['black', 'blue', 'green', 'red'], dtype=object)

In [87]:
print("\nClass mapping:")
for cls, idx in zip(le.classes_, range(len(le.classes_))):
    print(f"{cls} -> {idx}")

# Encode new data
new_val = ['blue','green']
encoded_val = le.transform(new_val)
print("\nNew value encoded:", encoded_val)

# Decode back (inverse transform)
decoded_val = le.inverse_transform(encoded_val)
print("Decoded back:", decoded_val)



Class mapping:
black -> 0
blue -> 1
green -> 2
red -> 3

New value encoded: [1 2]
Decoded back: ['blue' 'green']


In [88]:
# Append to dataframe
df1_new = pd.DataFrame({'color': new_val, 'color_encoded': encoded_val})
df1_final = pd.concat([df1, df1_new], ignore_index=True)
print(df1_final)

   color  color_encoded
0    red              3
1  green              2
2   blue              1
3  black              0
4    red              3
5   blue              1
6  green              2


#If it’s a brand-new category (like "yellow" that wasn’t seen in fit), 
# LabelEncoder will throw an error unless you refit.

~If it’s an already known color → just transform and append.

~If it’s an unknown color → you must refit on the new data.

In [89]:
# Add unseen category
new_value = ['yellow']
df1_extend=pd.DataFrame({'color':new_value})
df1_extended= pd.concat([df1,df1_extend], ignore_index=True)
# Refit encoder on full data
df1_extended['color_encoded'] = le.fit_transform(df1_extended['color'])
print(df1_extended)
print("\nNew class mapping:", dict(zip(le.classes_, range(len(le.classes_)))))


    color  color_encoded
0     red              3
1   green              2
2    blue              1
3   black              0
4     red              3
5  yellow              4

New class mapping: {'black': 0, 'blue': 1, 'green': 2, 'red': 3, 'yellow': 4}


==============================================================================================
Reusing case if anytime add new value

In [94]:
def add_new_value(df1, le, new_value,col='color',encoded_col='color_encoded'):
    new_df = df1.copy()
    if new_value in le.classes_:
        encoded=le.transform(new_value)
        df1_new=pd.DataFrame({col:new_value,encoded_col:encoded})
        new_df=pd.concat([new_df,df1_new],ignore_index=True)
        
    else:
        df1_extend=pd.DataFrame({col:new_value})
        df1_extended=pd.concat([new_df[[col]],df1_extend],ignore_index=True)
        df1_extended[encoded_col]=le.fit_transform(df1_extended[col])
        new_df=df1_extended
df1 = add_new_value(df1, le, ['blue'])
print("\nAfter adding known value 'blue':\n", df1)
        


AttributeError: 'NoneType' object has no attribute 'copy'

============================================================================================================

ORDINAL ENCODING
--RANKS
education_lvel:
high scl-1
clg-2
graduate-3

In [57]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
oe=OrdinalEncoder()
df1=pd.DataFrame({
    'size':['small','medium','large','medium','small']
})
df1

,size
0,small
1,medium
2,large
3,medium
4,small


In [56]:
#Create an instance of ordinal encoder and then fit_transform
encod=OrdinalEncoder(categories=[['small','medium','large']])
#you need to give less rank to small to high to large
df1['encoded_val']=encod.fit_transform(df1[['size']])
df1

,size,val,encoded_val
0,small,0.0,0.0
1,medium,1.0,1.0
2,large,2.0,2.0
3,medium,1.0,1.0
4,small,0.0,0.0


In [ ]:
#Transformimg for new data
print(encod.transform([['large']]))
df_new= pd.concat([df1, pd.DataFrame({'size': ['large']})], ignore_index=True)

[[2.]]


C:\Users\Varshitha\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but OrdinalEncoder was fitted with feature names
  warnings.warn(


=================================================================================================================

*TARGET GUIDED ORDINAL ENCODING*
~Encode categorical variable based on their relationship with the target variable.
It is useful when we have a categorical variable with a large no.of unique categories..
**Replace each categorical variable based on mean or medium of target variable of that category.
This creates monotonic relationship between categorical and target varaible -->improve predictive power of ur model.

In [61]:
df=pd.DataFrame({
    'city':['New York','London','Paris','Tokyo','New York','Paris'],
    'price':[200,150,300,250,180,320]
})
df

,city,price
0,New York,200
1,London,150
2,Paris,300
3,Tokyo,250
4,New York,180
5,Paris,320


In [64]:
mean_price=df.groupby('city')['price'].mean().to_dict()
mean_price

{'London': 150.0, 'New York': 190.0, 'Paris': 310.0, 'Tokyo': 250.0}

In [68]:
df['city_encoded']=df['city'].map(mean_price)
print(df)
df[['price','city_encoded']]

       city  price  city_encoded
0  New York    200         190.0
1    London    150         150.0
2     Paris    300         310.0
3     Tokyo    250         250.0
4  New York    180         190.0
5     Paris    320         310.0


,price,city_encoded
0,200,190.0
1,150,150.0
2,300,310.0
3,250,250.0
4,180,190.0
5,320,310.0


In [ ]:
import seaborn as sns
df=sns.load_dataset('tips')
df

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2


In [100]:
#convert time based on total_bill
mean_bill=df.groupby('time')['total_bill'].mean().to_dict()
print(mean_bill)
df['time_encoded']=df['time'].map(mean_bill)
df

{'Lunch': 17.168676470588235, 'Dinner': 20.79715909090909}


C:\Users\Varshitha\AppData\Local\Temp\ipykernel_11400\533634160.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  mean_bill=df.groupby('time')['total_bill'].mean().to_dict()


,total_bill,tip,sex,smoker,day,time,size,time_encoded
0,16.99,1.01,Female,No,Sun,Dinner,2,20.797159
1,10.34,1.66,Male,No,Sun,Dinner,3,20.797159
2,21.01,3.50,Male,No,Sun,Dinner,3,20.797159
3,23.68,3.31,Male,No,Sun,Dinner,2,20.797159
4,24.59,3.61,Female,No,Sun,Dinner,4,20.797159
...,...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3,20.797159
240,27.18,2.00,Female,Yes,Sat,Dinner,2,20.797159
241,22.67,2.00,Male,Yes,Sat,Dinner,2,20.797159
242,17.82,1.75,Male,No,Sat,Dinner,2,20.797159
